# Descriptive statistics

Statistical analysis of the stratdyn study data. Starts with demographic descriptive statistics across the `control` and `treatment` arms and the population as a whole, using `analysis/survey_data.csv` (see `analysis/README.md` for how that file was built).

In [1]:
import pandas as pd

survey = pd.read_csv("survey_data.csv", dtype=str)
survey.head()

,username,arm,session,gender,age,stem_education_years,english_proficiency,social_closeness,presurvey_trust_2,presurvey_risk_3,...,presurvey_control_2,postsurvey_control_2,postsurvey_risk_1,postsurvey_trust_3,postsurvey_risk_2,postsurvey_trust_1,postsurvey_control_3,postsurvey_trust_2,postsurvey_control_1,postsurvey_risk_3
0,user0011,control,01,Male,25,5,High,1,100,91,...,81,88,100,68,100,91,100,95,21,87
1,user0012,control,01,Male,26,5,High,1,54,64,...,67,67,64,23,63,67,62,69,87,75
2,user0013,control,01,Male,24,4,Medium-High,1,50,36,...,100,100,12,8,83,83,86,5,13,14
3,user0014,control,01,Male,27,5,High,1,67,43,...,55,68,34,58,58,70,50,57,68,27
4,user0015,treatment,01,Male,29,10,Fluent/Native,2,22,6,...,50,65,95,5,58,66,100,2,30,91


## Demographics: data cleaning

`analysis/survey_data.csv` already uses descriptive column names (see `analysis/README.md` and `results/README.md#demographics_csv` for the original raw codes and data quality notes), but a few still need coercion before they can be summarized numerically:

- **age** is free-text but always numeric in this data; coerced directly.
- **english_proficiency** is categorical (`Low` / `Medium-Low` / `Medium-High` / `High` / `Fluent/Native`). Mapped to an ordinal 1-5 scale so min/max/mean are meaningful. A couple of responses are the literal string `"undefined"` (the radio button was never selected) and become missing.
- **social_closeness** (how well a participant knew their partner) is already an ordinal 1-5 scale; a couple of `"undefined"` responses become missing here too.
- **gender** is `Female`/`Male` for every participant in this dataset (no `Other`/`Rather-not-say` responses were recorded).

Note: `stem_education_years` (education) is excluded from this analysis.

In [2]:
ENGLISH_PROFICIENCY_SCALE = {
    "Low": 1,
    "Medium-Low": 2,
    "Medium-High": 3,
    "High": 4,
    "Fluent/Native": 5,
}

# Keep the raw text before overwriting each column in place, so the
# exclusions check below can still tell "was missing" from "became missing".
raw_english_proficiency = survey["english_proficiency"].copy()
raw_social_closeness = survey["social_closeness"].copy()

survey["age"] = pd.to_numeric(survey["age"], errors="coerce")
survey["english_proficiency"] = raw_english_proficiency.map(ENGLISH_PROFICIENCY_SCALE)
survey["social_closeness"] = pd.to_numeric(raw_social_closeness, errors="coerce")

In [3]:
# How many non-missing raw responses became NaN after coercion, per arm --
# a transparency check so cleaning never silently drops data.
exclusion_checks = {
    "english_proficiency": (raw_english_proficiency, survey["english_proficiency"]),
    "social_closeness": (raw_social_closeness, survey["social_closeness"]),
}

excluded = pd.DataFrame({
    name: raw.notna() & clean.isna()
    for name, (raw, clean) in exclusion_checks.items()
})
excluded["arm"] = survey["arm"]
excluded.groupby("arm").sum()

,english_proficiency,social_closeness
arm,,
control,1,1
treatment,1,1


## Demographic summary table

Counts and min/mean/max by arm, plus an `overall` row for the population as a whole. `N` is the arm's total participant count; each item also reports its own sample size (`(N)`), since `english_proficiency` and `social_closeness` each have a couple of missing responses (see the exclusions check above) that make their own count smaller than the arm total.

In [4]:
def summarize_demographics(df):
    gender = df["gender"]
    return pd.Series({
        "N": len(df),
        "Female": (gender == "Female").sum(),
        "Male": (gender == "Male").sum(),
        "Age (N)": df["age"].count(),
        "Age (min)": df["age"].min(),
        "Age (mean)": df["age"].mean(),
        "Age (max)": df["age"].max(),
        "English proficiency (N)": df["english_proficiency"].count(),
        "English proficiency (min)": df["english_proficiency"].min(),
        "English proficiency (mean)": df["english_proficiency"].mean(),
        "English proficiency (max)": df["english_proficiency"].max(),
        "Social closeness (N)": df["social_closeness"].count(),
        "Social closeness (min)": df["social_closeness"].min(),
        "Social closeness (mean)": df["social_closeness"].mean(),
        "Social closeness (max)": df["social_closeness"].max(),
    })

by_arm = survey.groupby("arm").apply(summarize_demographics)
overall = summarize_demographics(survey).rename("overall")

demographics_summary = pd.concat([by_arm, overall.to_frame().T])
demographics_summary.round(2)

,N,Female,Male,Age (N),Age (min),Age (mean),Age (max),English proficiency (N),English proficiency (min),English proficiency (mean),English proficiency (max),Social closeness (N),Social closeness (min),Social closeness (mean),Social closeness (max)
control,24.0,7.0,17.0,24.0,21.0,27.04,38.0,23.0,3.0,4.26,5.0,23.0,1.0,1.65,4.0
treatment,28.0,8.0,20.0,28.0,20.0,25.32,33.0,27.0,2.0,4.15,5.0,27.0,1.0,2.22,5.0
overall,52.0,15.0,37.0,52.0,20.0,26.12,38.0,50.0,2.0,4.20,5.0,50.0,1.0,1.96,5.0


## Outcome descriptive statistics

Descriptive summary of task-round outcomes, using `analysis/task_data.csv` (see `analysis/README.md` for how that file was built; see `outcome_analysis.ipynb` for the full inferential analysis this only summarizes). For each round, both partners independently choose a strategy: `C` (collaborative) or `I` (individual), giving three possible outcomes:

- **Successful collaboration** -- both partners choose `C`.
- **Mutual independence** -- both partners choose `I`.
- **Coordination failure** -- one partner chooses `C` and the other `I`.

In [5]:
task = pd.read_csv("task_data.csv")

missing = (task["strategy_1"] == "undefined") | (task["strategy_2"] == "undefined")
print(f"Valid outcomes: {(~missing).sum()} of {len(task)} rounds "
      f"({missing.sum()} dropped -- undefined strategy, see results/README.md#task_csv--decision-task-rounds)")
task = task[~missing].copy()


def classify_outcome(row):
    s1, s2 = row["strategy_1"], row["strategy_2"]
    if s1 == "C" and s2 == "C":
        return "successful collaboration"
    if s1 == "I" and s2 == "I":
        return "mutual independence"
    return "coordination failure"


OUTCOME_ORDER = ["successful collaboration", "mutual independence", "coordination failure"]
task["outcome"] = task.apply(classify_outcome, axis=1)

counts = task.groupby("arm")["outcome"].value_counts().unstack(fill_value=0)[OUTCOME_ORDER]
counts.loc["overall"] = counts.sum()
proportions = counts.div(counts.sum(axis=1), axis=0) * 100

outcome_summary = pd.DataFrame(index=counts.index)
outcome_summary["N"] = counts.sum(axis=1)
for outcome in OUTCOME_ORDER:
    outcome_summary[f"{outcome} (n)"] = counts[outcome]
    outcome_summary[f"{outcome} (%)"] = proportions[outcome].round(1)

outcome_summary

Valid outcomes: 778 of 780 rounds (2 dropped -- undefined strategy, see results/README.md#task_csv--decision-task-rounds)


,N,successful collaboration (n),successful collaboration (%),mutual independence (n),mutual independence (%),coordination failure (n),coordination failure (%)
arm,,,,,,,
control,358,227,63.4,69,19.3,62,17.3
treatment,420,335,79.8,47,11.2,38,9.0
overall,778,562,72.2,116,14.9,100,12.9
